In [ ]:
import pickle
import subprocess
import pandas as pd
from tqdm import tqdm

In [ ]:
red_db_file  = "../2_compiling_unified_database_files/MICROSCAN_database.csv"
red_db = pd.read_csv(red_db_file)
red_db.head()

In [ ]:
filenames = list(red_db['File name'])

In [ ]:
#CNN1D

results = {}
for file_name in tqdm(filenames):
    # Define the command to be executed
    command = f"python -m src.infer -i ../1_converting_sp_to_csv/csv_files/{file_name} -m cnn1d_v0.2.0.onnx --model-name cnn -k 1"

    # Run the command and capture the output
    output = subprocess.check_output(command, shell=True).decode("utf-8")

    # Process the output to convert it into a list
    output_list = output.splitlines()

    results.update({file_name : output_list})

In [ ]:
results_red = {}
for i in tqdm(results.keys()):
    str_representation = eval(results[i][0])
    result_list = str_representation    
    results_red.update({i : result_list})

with open('cnn1d_results_full_db.pickle', 'wb') as file:
    pickle.dump(results_red, file)

In [ ]:
#import pickle
#
#with open('cnn1d_results_full_db.pickle', 'rb') as file:
#    results_red = pickle.load(file)

In [ ]:
# Transform the dictionary into a DataFrame
rows = []
for file_name, values in results_red.items():
    if len(values[0][0]) == 2 and values[0][0] == ['HDPE', 'LDPE']:
        rows.append({'File name': file_name, 'Polymer': 'PE', 'Matching': values[0][1], 'if_mixture': 'no'})
    elif len(values[0][0]) == 2:

        if values[0][0][0] == 'LDPE':
            pol1 = 'PE'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'LDPE':
            pol1 = 'PE'
            pol2 = values[0][0][0]
            
        elif values[0][0][0] == 'HDPE':
            pol1 = 'PE'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'HDPE':
            pol1 = 'PE'
            pol2 = values[0][0][0]
            
        elif values[0][0][0] == 'PP':
            pol1 = 'PP'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'PP':
            pol1 = 'PP'
            pol2 = values[0][0][0]
            
        elif values[0][0][0] == 'PS':
            pol1 = 'PS'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'PS':
            pol1 = 'PS'
            pol2 = values[0][0][0]
            
        elif values[0][0][0] == 'CPE':
            pol1 = 'CPE'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'CPE':
            pol1 = 'CPE'
            pol2 = values[0][0][0]
            
        else:
            pol1 = values[0][0][0]
            pol2 = values[0][0][1]
            
        rows.append({'File name': file_name, 'Polymer': pol1+'+'+pol2, 'Matching': values[0][1], 'if_mixture': 'yes'})
    elif len(values[0][0]) == 1 and values[0][0][0] == 'HDPE':
        rows.append({'File name': file_name, 'Polymer': 'PE', 'Matching': values[0][1], 'if_mixture': 'no'})
    elif len(values[0][0]) == 1 and values[0][0][0] == 'LDPE':
        rows.append({'File name': file_name, 'Polymer': 'PE', 'Matching': values[0][1], 'if_mixture': 'no'})
    elif len(values[0][0]) == 1:
        rows.append({'File name': file_name, 'Polymer': values[0][0][0], 'Matching': values[0][1], 'if_mixture': 'no'})

# Create a DataFrame
results_red_first = pd.DataFrame(rows)
results_red_first = results_red_first.merge(red_db[['File name']], on='File name', how='inner')

data = {'File name': results_red_first['File name'].to_list(), 'true' : red_db['Polymer'].to_list(), 'predicted' : results_red_first['Polymer'].to_list()}
df = pd.DataFrame(data)
df

In [ ]:
df.to_csv('manual_and_CNN1D_classification.csv', index=False)